# 14 — Alpha-rank LoRA: 静的・種類別の線形ランク配分

**本番未実行。** LLaMA-3.2-3B / Gemma3-4Bを1モデルずつ選択します。
元モデルを用い、DE・LRA・量子化・rsLoRAは適用しません。

- 同じモジュール種類（Q同士など）の中でalphaを比較。元重みの二乗特異値のESDを解析。
- $r_i^*=r_0(1+\beta z_i)$。$\beta=-0.5,0,+0.5$、$r_0=8$、seed 42が初期設定。
- 正はalpha大優先、負はalpha小優先。学習中はランクを固定。
- **各種類の追加パラメータ数を、条件間で均一ランク$r_0$と完全一致**させる。
  Qとupに同額を割り当てる、という意味ではない。
- 通常LoRA、`lora_alpha=2*r_i`、倍率2、dropout 0.05、biasなし。
- WikiText-2 raw train、BOSなし1024トークンの非重複完全ブロック、1 epoch。
- **今後の共通PPL評価は13と同じ `paper-2048-each-bos-v1`**：
  testを二重改行で連結し、特殊トークンなしでtokenize。
  BOS 1 + 本文2047、非重複、末尾不完全ブロック除外、本文を全て採点。
  BF16 / SDPA / batch 1 / TF32無効。testは設定選択に使わない。

これは2024年のLoRA-MoE用AlphaLoRAとは異なる実験です。
ここで使うピーク固定Hill型推定は、対数固有値ヒストグラムの最頻binの左端をxminとする
明示的な実装です。旧コードの自動フィットや、他実装のfix-fingerと数値一致を仮定しません。
alphaが不適切/推定不能なら中立配分とし、状態を必ず保存します。

実行順：環境 → 設定 → 関数 → 軽量検証 → 入力準備 → 本番 → 集計。
本番は`RUN_EXPERIMENTS=True`にしてから実行。外部の13ファイルは実行時に不要です。
モデルの利用許可とHF_TOKEN、CUDA GPU、十分なDrive容量が必要です。


In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = False  # 必要な場合True。完了後はランタイム再起動。
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'transformers==4.57.6', 'peft==0.21.0', 'accelerate>=1.0,<2',
                    'datasets>=3,<5', 'numpy>=1.26,<3', 'pandas>=2,<3',
                    'matplotlib>=3.8,<4', 'safetensors>=0.4'], check=True)
import importlib.metadata
for package, expected in {'transformers':'4.57.6', 'peft':'0.21.0'}.items():
    actual = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(package), '__version__', actual)
    if actual != expected or loaded != expected:
        raise RuntimeError(f'{package}=={expected}を導入してランタイムを再起動: {actual}, loaded={loaded}')
if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN (ログイン済みなら空欄): ').strip()
    if token: os.environ['HF_TOKEN'] = token


In [ ]:
import json, hashlib, math, gc, time, re, platform, inspect, shutil
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback, set_seed
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from torch.utils.data import Dataset

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    candidates = list((Path.home()/'Library/CloudStorage').glob('GoogleDrive-*/*/TUS/hashiguchi/data'))
    if len(candidates) != 1: raise RuntimeError('DATA_ROOTを手動設定してください: '+str(candidates))
    DATA_ROOT = candidates[0]

MODEL_KEY = 'llama'  # 'llama' / 'gemma'。必ず1モデルずつ実行。
MODELS = {'llama':'meta-llama/Llama-3.2-3B', 'gemma':'google/gemma-3-4b-pt'}
REVISIONS = {'llama':'13afe5124825b4f3751f836b40dafda64c1ed062',
             'gemma':'cc012e0a6d0787b4adcc0fa2c4da74402494554d'}
TOTAL = {'llama':3212749824, 'gemma':3880263168}  # Gemmaは言語部のみ
TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
REFERENCE_RANKS = [8]  # 予算を増やすなら [4,8,16] など
BETAS = [-0.5, 0.0, 0.5]
SEEDS = [42]
TRAIN = dict(epochs=1, learning_rate=2e-4, sequence_length=1024,
             batch=1, accumulation=8, dropout=0.05, alpha_per_rank=2,
             warmup_ratio=0.0, save_steps=100, logging_steps=10)
ALPHA = dict(version='log_esd_peak_hill_v1', bins=100, min_tail=50)
BATCH_SIZE = 1  # 評価用。13と同じ。
SVD_DEVICE = 'cuda'  # 推定時に1行列ずつFP32 svdvals。GPUメモリ制約時は'cpu'。
RUN_EXPERIMENTS = False
SAVE_CHECKPOINTS = True
OUTPUT_ROOT = DATA_ROOT/'alpha_rank_lora'/'static-linear-module-budget-v1'
OUTPUT = OUTPUT_ROOT/MODEL_KEY
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

assert MODEL_KEY in MODELS
assert TARGETS and len(TARGETS) == len(set(TARGETS))
assert REFERENCE_RANKS and all(isinstance(r,int) and r>=1 for r in REFERENCE_RANKS)
assert len(BETAS) == len(set(BETAS)) and all(abs(b)<1 for b in BETAS)
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert BATCH_SIZE == 1


## 共通PPL評価：13の処理をそのまま組み込み
以下の`evaluate`、`pack_paper_tokens`、`prepare_paper_inputs`は13からコピーした処理です。
lossはFP32で計算し、トークン数で重みづけしたNLLをFP64で集計します。
学習1024と評価2048は独立設定です。train/validation/testは混ぜません。
コード・データ・環境・入力token hashは実行結果のidentityに含めます。


In [ ]:
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def paper_json_default(x):
    if isinstance(x,(set,frozenset)): return sorted(x,key=repr)
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,(Path,torch.dtype,torch.device)):return str(x)
    raise TypeError(type(x).__name__)

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=paper_json_default));tmp.replace(path)

def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def pack_paper_tokens(raw, bos_id, length=2048):
    raw=np.asarray(raw,dtype=np.int64)
    if length<2 or bos_id is None: raise ValueError('Invalid BOS/length')
    n=len(raw)//(length-1)
    if not n: raise ValueError('No complete evaluation blocks')
    a=np.empty((n,length),dtype=np.int64);a[:,0]=bos_id
    a[:,1:]=raw[:n*(length-1)].reshape(n,length-1)
    return a,dict(blocks=n,length=length,scored_tokens=n*(length-1),dropped_tokens=len(raw)-n*(length-1),bos_token_id=int(bos_id))

def paper_environment():
    return dict(python=platform.python_version(),versions={k:importlib.metadata.version(k) for k in
        ('torch','transformers','datasets','numpy')},gpu=torch.cuda.get_device_name() if torch.cuda.is_available() else 'cpu',
        cuda=torch.version.cuda,attention='sdpa',tf32=False)

def prepare_paper_inputs(key, data_root, model_id, revision, train_length=None):
    from transformers import AutoTokenizer
    from datasets import load_dataset
    root=data_root/'ibis2026_de_lra/paper-2048-each-bos-inputs-v1';root.mkdir(parents=True,exist_ok=True)
    tok=None; results={}
    for split in (['test','train'] if train_length is not None else ['test']):
        length=2048 if split=='test' else train_length
        identity=dict(model=model_id,revision=revision,split=split,length=length,
            dataset_revision=PAPER_PROTOCOL['corpus_revision'],transformers=importlib.metadata.version('transformers'),
            packing=PAPER_PROTOCOL['packing'] if split=='test' else 'no BOS; disjoint full blocks; drop tail')
        p=root/f'{key}_{split}_{digest(identity)[:16]}.npy';receipt=p.with_suffix('.json')
        if p.exists() and receipt.exists():
            meta=json.loads(receipt.read_text())
            if meta['identity']!=identity or meta['sha256']!=sha(p):raise ValueError('Input cache checksum/identity mismatch')
            a=np.load(p,allow_pickle=False)
        else:
            text_path=root/f'{split}.txt'
            if not text_path.exists():
                old=data_root/'ibis2026_de_lra/gemma-qk-ppl-settings-grid-v1/inputs/wikitext2_test.txt'
                if split=='test' and old.exists():text=old.read_text(encoding='utf-8')
                else:
                    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split,revision=PAPER_PROTOCOL['corpus_revision'])
                    text='\n\n'.join(ds['text'])
                text_path.write_text(text,encoding='utf-8')
            if split=='test' and sha(text_path)!='696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83':
                raise ValueError('Test corpus differs from audited 11_1')
            if tok is None:tok=AutoTokenizer.from_pretrained(model_id,revision=revision,token=os.environ.get('HF_TOKEN'))
            raw=np.asarray(tok(text_path.read_text(encoding='utf-8'),add_special_tokens=False)['input_ids'],dtype=np.int64)
            if split=='test':a,meta=pack_paper_tokens(raw,tok.bos_token_id,length)
            else:
                a=raw[:len(raw)//length*length].reshape(-1,length).copy()
                if not len(a):raise ValueError('Empty training data')
                meta=dict(blocks=len(a),length=length,dropped_tokens=len(raw)%length)
            with p.with_suffix('.tmp').open('wb') as f:np.save(f,a,allow_pickle=False)
            p.with_suffix('.tmp').replace(p)
            meta.update(identity=identity,sha256=sha(p),text_sha256=sha(text_path));write_json(receipt,meta)
        if a.ndim!=2 or a.shape[1]!=length or not len(a):raise ValueError('Token shape mismatch')
        if split=='test' and not np.all(a[:,0]==meta['bos_token_id']):raise ValueError('BOS missing')
        results[split]=(a,meta)
    return results

def evaluate(model,tokens):
    model.eval()
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
        if start==0 or (start+1)%25==0 or start+1==len(tokens): print(f'  PPL blocks {start+1}/{len(tokens)}',flush=True)
    return np.asarray(losses),np.asarray(counts)

evaluate = torch.inference_mode()(evaluate)
PAPER_PROTOCOL = dict(version='paper-2048-each-bos-v1', length=2048, bos='each_block',
    packing='one BOS + 2047 text; disjoint; drop incomplete tail', batch=1,
    corpus_revision='b08601e04326c79dfdd32d625aee71d232d685c3', precision='bf16', attention='sdpa')

def release():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def atomic_npy(path, array):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    with tmp.open('wb') as f: np.save(f, array, allow_pickle=False)
    tmp.replace(path)

def environment():
    env = paper_environment()
    env['versions'].update({k:importlib.metadata.version(k) for k in ['peft','accelerate','safetensors']})
    return env

def tensor_hash(w):
    return hashlib.sha256(w.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes()).hexdigest()

def model_load():
    if not torch.cuda.is_available(): raise RuntimeError('本番実験にはCUDA GPUが必要です')
    kwargs = dict(revision=REVISIONS[MODEL_KEY], token=os.environ.get('HF_TOKEN'),
                  torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                  output_loading_info=True, attn_implementation='sdpa')
    if MODEL_KEY == 'gemma':
        from transformers import Gemma3ForConditionalGeneration
        full, info = Gemma3ForConditionalGeneration.from_pretrained(MODELS[MODEL_KEY], **kwargs)
        check_loading_info(info)
        model = extract_gemma_text(full)
        del full; release()
    else:
        model, info = AutoModelForCausalLM.from_pretrained(MODELS[MODEL_KEY], **kwargs)
        check_loading_info(info)
    count = sum(p.numel() for p in model.parameters())
    if count != TOTAL[MODEL_KEY]: raise ValueError(f'Base parameter count: {count}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded meta tensors')
    model.config.use_cache = False
    return model.to('cuda').eval()


## alpha解析と静的配分
種類$t$について、$c_i=m_i+n_i$、重み付き平均$\bar\alpha_t=\sum c_i\alpha_i/\sum c_i$、
$z_i=(\alpha_i-\bar\alpha_t)/\max_j|\alpha_j-\bar\alpha_t|$、
$r_i^*=r_0(1+\beta z_i)$。
推定不能な行列は中心値（$z_i=0$）として扱います。
現在の2モデルでは同じ種類の行列形状が共通なので、整数化は**種類別のランク合計を厳密に維持**します。
floor後の不足分を、二乗丸め誤差の増分が小さい順に1ずつ配ります。
同点は完全なモジュール名で決定。整数化後は線形から丸め分だけずれます。
同じ種類内で形状が異なるモデルは、黙って近似せずエラーにします。


In [ ]:
def matrix_inventory(model):
    rows = []
    for name, module in model.named_modules():
        kind = name.rsplit('.',1)[-1]
        if kind not in TARGETS: continue
        if not isinstance(module, torch.nn.Linear): raise TypeError(f'Expected Linear: {name}')
        if not re.search(r'(?:^|\.)layers\.\d+\.', name):
            raise ValueError(f'Unexpected non-decoder target: {name}')
        m,n = module.weight.shape
        rows.append(dict(name=name, kind=kind, m=m, n=n, cost=m+n))
    df = pd.DataFrame(rows).sort_values('name').reset_index(drop=True)
    if not len(df) or set(df.kind) != set(TARGETS): raise ValueError('Missing target module type')
    if df.name.duplicated().any(): raise ValueError('Duplicate full module name')
    layers = set(re.findall(r'layers\.(\d+)\.', ' '.join(df.name)))
    expected_layers = set(map(str,range(model.config.num_hidden_layers)))
    if layers != expected_layers: raise ValueError('Incomplete decoder layer coverage')
    for kind, group in df.groupby('kind'):
        if len(group) != len(expected_layers): raise ValueError(f'Incomplete {kind}')
        if group.cost.nunique() != 1 or len(group[['m','n']].drop_duplicates()) != 1:
            raise ValueError(f'Mixed shapes within {kind}: exact-cost integer allocator must be extended')
    return df

def fit_alpha(eigenvalues):
    x = np.sort(np.asarray(eigenvalues,dtype=np.float64))
    x = x[np.isfinite(x) & (x>0)]
    empty = dict(alpha=None, status='insufficient_tail', xmin=None, n_tail=0, ks=None)
    if len(x) < ALPHA['min_tail']: return empty
    logs = np.log10(x)
    hist, edges = np.histogram(logs, bins=ALPHA['bins'])
    xmin = float(10**edges[int(np.argmax(hist))])
    tail = x[x>=xmin]
    if len(tail) < ALPHA['min_tail']: return dict(empty,xmin=xmin,n_tail=len(tail))
    den = np.log(tail/xmin).sum()
    if not np.isfinite(den) or den<=0: return dict(empty,status='degenerate_tail',xmin=xmin,n_tail=len(tail))
    alpha = float(1+len(tail)/den)
    cdf = 1-(tail/xmin)**(1-alpha)
    empirical = np.arange(1,len(tail)+1)/len(tail)
    ks = float(max(np.max(np.abs(cdf-empirical)),np.max(np.abs(cdf-(empirical-1/len(tail))))))
    return dict(alpha=alpha,status='ok',xmin=xmin,n_tail=len(tail),ks=ks)

def analyze_alpha(model, inventory):
    rows=[]
    for index, row in inventory.iterrows():
        row = row.to_dict()
        weight = model.get_submodule(row['name']).weight
        identity = dict(model=MODELS[MODEL_KEY], revision=REVISIONS[MODEL_KEY],
                        weight_sha256=tensor_hash(weight), name=row['name'], shape=list(weight.shape),
                        estimator=ALPHA, estimator_code=ALPHA_CODE_HASH,
                        torch=torch.__version__, svd_device=SVD_DEVICE)
        root=OUTPUT/'spectra'; root.mkdir(parents=True,exist_ok=True)
        p=root/(digest(identity)+'.npy'); receipt=p.with_suffix('.json')
        if p.exists() and receipt.exists():
            saved=json.loads(receipt.read_text())
            if saved['identity']!=identity or sha(p)!=saved['sha256']: raise ValueError('Spectral cache mismatch')
            fit=saved['fit']
        else:
            start=time.perf_counter()
            w=weight.detach().to(device=SVD_DEVICE,dtype=torch.float32)
            singular=torch.linalg.svdvals(w)
            if not torch.isfinite(singular).all(): raise FloatingPointError('Invalid SVD')
            eig=singular.double().square().cpu().numpy()
            del w,singular; release()
            fit=fit_alpha(eig)
            atomic_npy(p,eig)
            write_json(receipt,dict(identity=identity,sha256=sha(p),fit=fit,seconds=time.perf_counter()-start))
        rows.append(dict(row,**fit,weight_sha256=identity['weight_sha256'],spectrum_sha256=sha(p)))
        print(f"alpha {index+1}/{len(inventory)} {row['name']}: {fit['alpha']} ({fit['status']})",flush=True)
    result=pd.DataFrame(rows)
    return result

def rank_plan(stats, r0, beta):
    if abs(beta)>=1 or r0<1: raise ValueError('Require r0>=1, |beta|<1')
    frames=[]
    for kind, group in stats.groupby('kind',sort=True):
        group=group.sort_values('name').copy()
        if group.cost.nunique()!=1: raise ValueError('Mixed per-rank costs within module type')
        cost=int(group.cost.iloc[0]); capacity=np.minimum(group.m,group.n).to_numpy(dtype=int)
        valid=(group.status=='ok').to_numpy() & np.isfinite(pd.to_numeric(group.alpha,errors='coerce').to_numpy(dtype=float))
        raw=pd.to_numeric(group.alpha,errors='coerce').to_numpy(dtype=float)
        mean=float(np.average(raw[valid],weights=group.cost.to_numpy()[valid])) if valid.any() else 0.0
        centered=np.where(valid,raw-mean,0.0)
        scale=float(np.max(np.abs(centered)))
        z=centered/scale if scale>0 else np.zeros(len(group))
        target=r0*(1+beta*z)
        low=max(1,math.floor(r0*(1-abs(beta))))
        high=math.ceil(r0*(1+abs(beta)))
        if np.any(target<1-1e-8) or np.any(target>capacity+1e-8):
            raise ValueError('Requested budget/slope violates valid rank bounds')
        ranks=np.floor(target+1e-10).astype(int)
        remainder=int(r0*len(group)-int(ranks.sum()))
        if not 0<=remainder<=len(group): raise ValueError('Invalid remainder')
        # Incremental squared error: (floor+1-target)^2 - (floor-target)^2.
        candidates=sorted(range(len(group)),key=lambda j:(1-2*(target[j]-ranks[j]),str(group.name.iloc[j])))
        for j in candidates[:remainder]: ranks[j]+=1
        if np.any(ranks<low) or np.any(ranks>np.minimum(high,capacity)): raise ValueError('Rounded rank out of bounds')
        if int(ranks.sum())!=r0*len(group): raise AssertionError('Per-kind rank budget mismatch')
        group['alpha_center']=mean;group['z']=z;group['target_rank']=target
        group['rank']=ranks;group['lora_alpha']=2*ranks
        group['added_parameters']=cost*ranks
        group['reference_parameters']=cost*r0
        group['reference_rank']=r0;group['beta']=float(beta)
        if int(group.added_parameters.sum())!=int(group.reference_parameters.sum()):
            raise AssertionError('Per-kind parameter budget mismatch')
        frames.append(group)
    plan=pd.concat(frames).sort_values('name').reset_index(drop=True)
    if not plan.name.is_unique: raise AssertionError('Duplicate plan names')
    return plan

def attach_lora(model, plan):
    rank_pattern={str(row['name']):int(row['rank']) for row in plan.to_dict('records')}
    alpha_pattern={str(row['name']):int(row['lora_alpha']) for row in plan.to_dict('records')}
    if set(rank_pattern)!=set(matrix_inventory(model).name): raise ValueError('Target/plan mismatch')
    config=LoraConfig(task_type=TaskType.CAUSAL_LM, r=1, lora_alpha=2,
        rank_pattern=rank_pattern, alpha_pattern=alpha_pattern,
        # Full, escaped module paths: no suffix collisions or accidental vision adapters.
        target_modules='^(?:'+'|'.join(re.escape(name) for name in rank_pattern)+')$',
        lora_dropout=TRAIN['dropout'],bias='none',use_rslora=False,init_lora_weights=True)
    model=get_peft_model(model,config)
    verify_lora(model,plan)
    return model

def verify_lora(model, plan):
    expected={row['name']:row for row in plan.to_dict('records')};seen={}
    prefix='base_model.model.'
    for name,module in model.named_modules():
        if not hasattr(module,'lora_A') or 'default' not in module.lora_A: continue
        key=name[len(prefix):] if name.startswith(prefix) else name
        if key not in expected or key in seen: raise AssertionError(f'Unexpected/duplicate adapter: {name}')
        row=expected[key];rank=int(row['rank'])
        if module.r['default']!=rank or module.lora_A['default'].weight.shape!=(rank,int(row['n'])):
            raise AssertionError(f'LoRA rank mismatch: {name}')
        if module.lora_B['default'].weight.shape!=(int(row['m']),rank): raise AssertionError('B shape mismatch')
        if module.lora_alpha['default']!=2*rank or not math.isclose(module.scaling['default'],2.0):
            raise AssertionError('LoRA scaling mismatch')
        seen[key]=rank
    if set(seen)!=set(expected): raise AssertionError('Missing adapters')
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable!=int(plan.added_parameters.sum()): raise AssertionError('Actual trainable budget mismatch')
    return trainable


## 保存・再開と学習
条件ごとにモデルを元checkpointから再ロードするので、別条件の更新を引き継ぎません。
完全なチェックサムreceiptがあるcheckpointのみ再開します。設定・環境・入力・コードの変更は別identity。
完成adapterと評価もchecksumを照合して再利用します。GPUエラー等はその場で停止し、成功扱いしません。


In [ ]:
class PackedTokens(Dataset):
    def __init__(self,array): self.array=array
    def __len__(self): return len(self.array)
    def __getitem__(self,i):
        ids=torch.as_tensor(self.array[i].copy(),dtype=torch.long)
        return dict(input_ids=ids,attention_mask=torch.ones_like(ids),labels=ids.clone())

def snapshot(root):
    return {str(p.relative_to(root)):sha(p) for p in sorted(root.rglob('*'))
            if p.is_file() and p.name!='receipt.json'}

def verify_snapshot(root, files):
    if not files: raise ValueError('Empty receipt')
    for rel, checksum in files.items():
        p=root/rel
        if not p.is_file() or sha(p)!=checksum: raise ValueError(f'Corrupt/incomplete file: {p}')

def read_receipt(root, identity):
    p=root/'receipt.json'
    if not p.exists(): return None
    result=json.loads(p.read_text())
    if result['identity']!=identity: raise ValueError('Saved identity mismatch')
    verify_snapshot(root,result['files'])
    return result

class CompleteCheckpoint(TrainerCallback):
    def __init__(self,identity): self.identity=identity
    def on_save(self,args,state,control,**kwargs):
        root=Path(args.output_dir)/f'checkpoint-{state.global_step}'
        write_json(root/'receipt.json',dict(identity=self.identity,step=state.global_step,files=snapshot(root)))

def resume_checkpoint(path,identity):
    roots=sorted((path/'trainer').glob('checkpoint-*'),key=lambda p:int(p.name.split('-')[-1]),reverse=True)
    for root in roots:
        try:
            if read_receipt(root,identity): return str(root)
        except ValueError as error: print('Ignoring invalid checkpoint:',root.name,str(error),flush=True)
    return None

def save_evaluation(model, tokens, root, identity, extra):
    previous=read_receipt(root,identity)
    if previous: return json.loads((root/'result.json').read_text())
    root.mkdir(parents=True,exist_ok=True)
    losses,counts=evaluate(model,tokens)
    total=float(losses.sum()); count=int(counts.sum()); mean=total/count
    result=dict(identity=identity,**extra,nll_sum=total,scored_tokens=count,mean_nll=mean,
                ppl=float(math.exp(mean)) if mean<709 else None,
                ppl_overflow=bool(mean>=709),blocks=len(losses))
    atomic_npy(root/'nll_sums.npy',losses);atomic_npy(root/'token_counts.npy',counts)
    write_json(root/'result.json',result)
    write_json(root/'receipt.json',dict(identity=identity,files=snapshot(root)))
    return result

def train_condition(model,plan,path,identity,seed):
    final=path/'adapter'
    receipt=read_receipt(final,identity)
    if receipt:
        print('SKIP training: verified saved adapter',flush=True)
        peft=PeftModel.from_pretrained(model,str(final),is_trainable=True)
        verify_lora(peft,plan)
        return peft
    set_seed(seed)
    model=attach_lora(model,plan)
    model.config.use_cache=False
    model.enable_input_require_grads()
    args=TrainingArguments(output_dir=str(path/'trainer'),num_train_epochs=TRAIN['epochs'],
        learning_rate=TRAIN['learning_rate'],per_device_train_batch_size=TRAIN['batch'],
        gradient_accumulation_steps=TRAIN['accumulation'],warmup_ratio=TRAIN['warmup_ratio'],
        lr_scheduler_type='linear',weight_decay=0.0,optim='adamw_torch',bf16=True,fp16=False,
        gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
        save_strategy='steps' if SAVE_CHECKPOINTS else 'no',save_steps=TRAIN['save_steps'],save_total_limit=2,
        logging_steps=TRAIN['logging_steps'],report_to='none',dataloader_num_workers=0,
        dataloader_pin_memory=True,seed=seed,data_seed=seed,remove_unused_columns=False,
        tf32=False)
    trainer=Trainer(model=model,args=args,train_dataset=PackedTokens(TRAIN_TOKENS),
                    callbacks=[CompleteCheckpoint(identity)])
    start=time.perf_counter()
    output=trainer.train(resume_from_checkpoint=resume_checkpoint(path,identity))
    write_json(path/'training_metrics.json',dict(metrics=output.metrics,wall_seconds=time.perf_counter()-start,
                                               log_history=trainer.state.log_history))
    temporary=path/'adapter.tmp'
    if temporary.exists(): shutil.rmtree(temporary)
    trainer.model.save_pretrained(temporary,safe_serialization=True)
    write_json(temporary/'receipt.json',dict(identity=identity,files=snapshot(temporary)))
    if final.exists(): raise RuntimeError(f'Unexpected incomplete final adapter: {final}')
    temporary.rename(final)
    model=trainer.model
    model.gradient_checkpointing_disable()
    del trainer
    return model.eval()

def validate_fresh_weights(model,stats):
    for row in stats.to_dict('records'):
        if tensor_hash(model.get_submodule(row['name']).weight)!=row['weight_sha256']:
            raise ValueError(f'Base checkpoint weights differ: {row["name"]}')

def run_all():
    if not RUN_EXPERIMENTS:
        print('本番は未実行です。設定確認後 RUN_EXPERIMENTS=True にしてください。');return
    if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU required')
    if 'INPUT_IDENTITY' not in globals(): raise RuntimeError('先に入力準備セルを実行してください')
    print('Original model loading / alpha analysis',flush=True)
    base=model_load()
    inventory=matrix_inventory(base)
    started=time.perf_counter()
    stats=analyze_alpha(base,inventory)
    spectral_seconds=time.perf_counter()-started
    spec_identity=digest(stats.to_json(orient='records',double_precision=15))
    stats_path=OUTPUT/'analysis'/spec_identity
    stats_path.mkdir(parents=True,exist_ok=True)
    stats.to_csv(stats_path/'alpha.csv',index=False)
    write_json(stats_path/'summary.json',dict(estimator=ALPHA,identity=spec_identity,seconds=spectral_seconds,
        statuses=stats.status.value_counts().to_dict(),note='seconds includes cache lookup on rerun; individual SVD time in spectral receipts'))
    base_identity=dict(common=INPUT_IDENTITY,kind='original_baseline')
    baseline=save_evaluation(base,TEST_TOKENS,OUTPUT/'baseline'/digest(base_identity)[:20],base_identity,
                             dict(model=MODEL_KEY,condition='original',added_parameters=0))
    del base; release()
    records=[]
    for r0 in REFERENCE_RANKS:
        for beta in BETAS:
            plan=rank_plan(stats,r0,beta)
            plan_records=json.loads(plan.to_json(orient='records',double_precision=15))
            plan_hash=digest(plan_records)
            plan_root=OUTPUT/'plans'/plan_hash
            plan_root.mkdir(parents=True,exist_ok=True)
            plan.to_csv(plan_root/'rank_plan.csv',index=False)
            budgets=plan.groupby('kind')[['added_parameters','reference_parameters']].sum()
            budgets.to_csv(plan_root/'module_budgets.csv')
            write_json(plan_root/'plan.json',dict(records=plan_records,sha256=sha(plan_root/'rank_plan.csv')))
            print(f'r0={r0}, beta={beta:+g}: added={int(plan.added_parameters.sum()):,}',flush=True)
            display(budgets)
            for seed in SEEDS:
                identity=dict(common=INPUT_IDENTITY,kind='alpha_rank_lora',plan_hash=plan_hash,
                              reference_rank=r0,beta=float(beta),seed=seed,spectral_identity=spec_identity,
                              train=TRAIN,checkpoint_saving=SAVE_CHECKPOINTS)
                path=OUTPUT/'runs'/digest(identity)[:24]
                path.mkdir(parents=True,exist_ok=True)
                write_json(path/'identity.json',identity)
                extra=dict(model=MODEL_KEY,reference_rank=r0,beta=float(beta),seed=seed,
                           added_parameters=int(plan.added_parameters.sum()),
                           added_fraction=int(plan.added_parameters.sum())/TOTAL[MODEL_KEY],plan_hash=plan_hash)
                if read_receipt(path/'evaluation',identity):
                    result=json.loads((path/'evaluation/result.json').read_text())
                    print('SKIP complete result:',path.name,flush=True)
                else:
                    set_seed(seed);model=model_load()
                    try:
                        validate_fresh_weights(model,stats)
                        model=train_condition(model,plan,path,identity,seed)
                        # Match 13: merge LoRA into BF16 base weights before PPL evaluation.
                        model=model.merge_and_unload()
                        model.gradient_checkpointing_disable();model.config.use_cache=False
                        result=save_evaluation(model,TEST_TOKENS,path/'evaluation',identity,extra)
                    finally:
                        del model;release()
                records.append({k:v for k,v in result.items() if k!='identity'})
                pd.DataFrame(records).to_csv(OUTPUT/'current_results.csv',index=False)
    print('All requested conditions completed. Results:',OUTPUT,flush=True)
    return pd.DataFrame(records)


## 軽量検証（モデルのダウンロード・学習なし）
予算・符号・中立配分・整数化を確認します。本番の実際のLoRA形状・倍率・学習対象数は、
adapter作成直後に必ず再検証します。


In [ ]:
def smoke_test():
    raw=np.arange(15,dtype=np.int64)
    a,meta=pack_paper_tokens(raw,99,length=5)
    assert a.shape==(3,5) and np.all(a[:,0]==99) and meta['scored_tokens']==12
    assert np.array_equal(a[:,1:].flatten(),raw[:12]) and meta['dropped_tokens']==3
    rows=[]
    for kind,cost,m,n in [('q_proj',64,32,32),('up_proj',96,64,32)]:
        for i,alpha in enumerate([2.,3.,4.,5.,6.]):
            rows.append(dict(name=f'model.layers.{i}.{kind}',kind=kind,m=m,n=n,cost=cost,alpha=alpha,status='ok'))
    stats=pd.DataFrame(rows)
    for r0 in [4,8,16]:
        for beta in [-0.5,0.,0.5]:
            p=rank_plan(stats,r0,beta)
            for kind,g in p.groupby('kind'):
                assert g.added_parameters.sum()==g.reference_parameters.sum()
                if beta==0: assert (g['rank']==r0).all()
                else: assert np.all(np.diff(g.sort_values('alpha')['rank'])*np.sign(beta)>=0)
    bad=stats.copy();bad['alpha']=np.nan;bad['status']='insufficient_tail'
    assert (rank_plan(bad,8,0.5)['rank']==8).all()
    same=stats.copy();same['alpha']=3.
    assert (rank_plan(same,8,-0.5)['rank']==8).all()
    fit=fit_alpha((1-(np.arange(200)+0.5)/200)**(-0.5))
    assert fit['status']=='ok' and fit['alpha']>1
    print('PASS: packing, exact module budgets, signed allocations, invalid/tied alpha fallback')

smoke_test()


## 入力準備・実行identity
このセルはモデル重みをロードしません。13の共通入力cacheをチェックサム付きで利用します。
以後の条件変更時は設定からこのセルまで再実行してください。


In [ ]:
def code_fingerprints():
    # Notebook内の関数コードをハッシュ。保存・評価・予算設定変更時に古い結果を混ぜない。
    names=['matrix_inventory','fit_alpha','analyze_alpha','rank_plan','attach_lora','verify_lora',
           'model_load','extract_gemma_text','evaluate','pack_paper_tokens','prepare_paper_inputs',
           'train_condition','save_evaluation','run_all','validate_fresh_weights',
           'read_receipt','resume_checkpoint','snapshot','verify_snapshot']
    hashes={}
    for name in names:
        obj=globals()[name]
        obj=inspect.unwrap(obj)
        hashes[name]=hashlib.sha256(inspect.getsource(obj).encode()).hexdigest()
    for cls,methods in [(PackedTokens,['__init__','__len__','__getitem__']), (CompleteCheckpoint,['__init__','on_save'])]:
        for method in methods:
            hashes[cls.__name__+'.'+method]=hashlib.sha256(inspect.getsource(getattr(cls,method)).encode()).hexdigest()
    return hashes

CODE_FINGERPRINTS=code_fingerprints()
ALPHA_CODE_HASH=digest({k:CODE_FINGERPRINTS[k] for k in ['fit_alpha','analyze_alpha']})
INPUTS=prepare_paper_inputs(MODEL_KEY,DATA_ROOT,MODELS[MODEL_KEY],REVISIONS[MODEL_KEY],train_length=TRAIN['sequence_length'])
TRAIN_TOKENS,TRAIN_META=INPUTS['train']
TEST_TOKENS,TEST_META=INPUTS['test']
INPUT_IDENTITY=dict(model=MODELS[MODEL_KEY],revision=REVISIONS[MODEL_KEY],protocol=PAPER_PROTOCOL,
                    train_tokens=TRAIN_META,test_tokens=TEST_META,environment=environment(),
                    code=CODE_FINGERPRINTS,alpha=ALPHA,targets=TARGETS,algorithm='Alpha-rank LoRA/static-linear-module-budget-v1')
write_json(OUTPUT/'latest_input_identity.json',INPUT_IDENTITY)
display(pd.DataFrame([dict(split='train',**{k:TRAIN_META[k] for k in ['blocks','length','dropped_tokens']}),
                      dict(split='test',**{k:TEST_META[k] for k in ['blocks','length','dropped_tokens','scored_tokens']})]))
print('Conditions:',len(REFERENCE_RANKS)*len(BETAS)*len(SEEDS),'for',MODEL_KEY)


## 本番（1モデルずつ）
設定セルの`RUN_EXPERIMENTS=True`にして、入力準備まで再実行してから次のセルを実行します。
各種類の予算・alphaとrankの対応表は`plans/`に保存されます。
このノートブックは自動切断しません。


In [ ]:
RESULTS = run_all()


## 集計と可視化
設定一致の現行runだけを表示。seedを増やした場合は平均・標準偏差を表示します。
1 seedの結果だけで配分方向を確定せず、test PPLでハイパーパラメータを選択しません。
追加パラメータの削減率と、モデル全体のメモリ/時間削減率は区別してください。


In [ ]:
if RESULTS is not None and len(RESULTS):
    display(RESULTS[['model','reference_rank','beta','seed','added_parameters','mean_nll','ppl']])
    display(RESULTS.groupby(['reference_rank','beta']).agg(
        seeds=('seed','count'),ppl_mean=('ppl','mean'),ppl_std=('ppl','std'),
        nll_mean=('mean_nll','mean'),added_parameters=('added_parameters','first')).reset_index())
    for r0,g in RESULTS.groupby('reference_rank'):
        fig,ax=plt.subplots(figsize=(6,3))
        for seed,data in g.groupby('seed'):
            data=data.sort_values('beta');ax.plot(data.beta,data.ppl,'o-',label=f'seed {seed}')
        ax.set(xlabel='Linear allocation slope beta',ylabel='WikiText-2 test PPL',title=f'{MODEL_KEY}: rank-{r0}-equivalent budget')
        ax.legend();fig.tight_layout();plt.show()
else:
    print('本番結果はまだありません。軽量検証のみでは学習・評価結果は生成しません。')
